# FXGuard AI - Current Project Overview

FXGuard AI is a full-stack exchange-rate risk classification and payment-planning system for Rwanda-based importers. The current application assesses short-term depreciation pressure for **USD/RWF, EUR/RWF, and KES/RWF** using official National Bank of Rwanda (BNR) exchange-rate histories.

For each supported currency, the system provides separate **7-day** and **14-day** Low, Medium, or High risk classifications. It combines the model result with the selected payment amount to show the current RWF cost, possible extra cost, a planning-buffer estimate, probability distribution, recent rate signals, and plain-language payment considerations.

> FXGuard AI is a research prototype and decision-support tool. Its classifications and estimates are not financial advice and do not guarantee future exchange rates.

## What is implemented now

- Multi-currency dashboard and payment checks for USD, EUR, and KES against RWF
- Six trained classifiers: one model for every currency and forecast horizon
- Official BNR buying, average, and selling rates imported from local Excel exports
- Historical trend charts, data-freshness reporting, and model metadata through the API
- Low, Medium, and High risk results with class probabilities and recent-rate drivers
- RWF payment-cost, possible-extra-cost, and planning-buffer estimates
- Excel export of an assessment result
- Optional email or phone one-time-code accounts through Supabase
- Guest device history and authenticated, user-owned cloud history
- Check deletion, data export, account deletion, consent records, Privacy Notice, and Terms of Use
- Embedded participant feedback form, with local feedback endpoints retained as a backup
- Render deployment for a static frontend and a FastAPI service

## Current architecture

| Layer | Current implementation |
|---|---|
| Data source | Official BNR Excel exports for USD, EUR, and KES |
| Data pipeline | Validation, daily-calendar alignment, feature engineering, and horizon labels in `scripts/sync_multicurrency_rates.py` |
| Model pipeline | Logistic regression, random forest, and XGBoost candidates in `scripts/train_multicurrency_models.py` |
| Evaluation | Three expanding-window rolling-origin folds, horizon purge gaps, and a final untouched 20% holdout |
| Model artifacts | Six joblib models and `backend/models/multicurrency_model_metadata.json` |
| API | FastAPI routes for rates, history, predictions, exports, feedback, authentication, and saved checks |
| Frontend | Responsive HTML/CSS/JavaScript dashboard, assessment, results, decision support, feedback, account, and legal views |
| Account storage | Supabase Auth and PostgreSQL with row-level access policies; guest use remains available |
| Deployment | Render static site plus Render Python web service; the API also serves a fallback frontend |

In [ ]:
from pathlib import Path
import json
import pandas as pd

# This works when Jupyter starts in either the project root or notebooks/.
ROOT = Path.cwd().resolve()
if not (ROOT / 'backend').exists() and (ROOT.parent / 'backend').exists():
    ROOT = ROOT.parent

expected_paths = [
    'backend/app',
    'backend/models',
    'data/raw',
    'data/processed',
    'frontend',
    'scripts',
    'supabase/migrations',
    'tests',
    'reports',
]

print('Project root:', ROOT)
print('\nCurrent project components:')
for relative_path in expected_paths:
    status = 'present' if (ROOT / relative_path).exists() else 'missing'
    print(f'  {relative_path:<24} {status}')

## Data and model scope

The active pipeline uses the average BNR rate as `mid_rate` and retains the official buying and selling rates. It engineers returns, moving averages, volatility, momentum, spread, and recent depreciation-day features. Currency-specific thresholds convert future depreciation into Low, Medium, and High labels.

Model selection compares logistic regression, random forest, and XGBoost by mean rolling-origin balanced accuracy, then macro F1 and accuracy. A 7-row or 14-row purge gap separates training and evaluation windows so future-derived labels do not cross a split boundary. The selected production model is refitted on all available labelled rows after evaluation.

The next cell reads the current metadata artifact, so its output reflects the models and BNR data bundled with this checkout.

In [ ]:
metadata_path = ROOT / 'backend' / 'models' / 'multicurrency_model_metadata.json'
with metadata_path.open(encoding='utf-8') as handle:
    metadata = json.load(handle)

coverage_rows = []
for currency in metadata['currencies']:
    coverage = metadata['data']['coverage'][currency]
    coverage_rows.append({
        'currency': currency,
        'first_official_date': coverage['first_date'],
        'latest_official_date': coverage['latest_date'],
        'official_observations': coverage['official_observations'],
    })

print('Official BNR data coverage')
display(pd.DataFrame(coverage_rows))

model_rows = []
for currency, horizons in metadata['models'].items():
    for horizon, details in horizons.items():
        model_rows.append({
            'currency': currency,
            'horizon': horizon,
            'selected_model': details['best_model'],
            'deployment_rows': details['deployment_training_rows'],
            'backtest_folds': details['backtest']['fold_count'],
            'purge_gap_rows': details['purge_gap_rows'],
            'model_file': details['model_file'],
        })

print('\nActive production models')
display(pd.DataFrame(model_rows))

In [ ]:
processed = ROOT / 'data' / 'processed'
dataset_rows = []
for file in sorted(processed.glob('multicurrency*.csv')):
    frame = pd.read_csv(file)
    dataset_rows.append({
        'dataset': file.name,
        'rows': len(frame),
        'columns': len(frame.columns),
    })

print('Current multicurrency processed datasets')
display(pd.DataFrame(dataset_rows))

## Main user and API flows

1. The dashboard loads supported currencies, recent official rates, a trend chart, and data freshness.
2. A user selects USD, EUR, or KES, enters a foreign-currency payment amount, and chooses 7 or 14 days.
3. `POST /api/predict-risk` loads the matching currency/horizon model and returns the risk class, probabilities, payment-cost estimates, recent signals, and considerations.
4. The result can be exported to Excel. A guest can keep it on the device, or a verified account can save it to user-owned cloud history.
5. Authenticated users can review or delete saved checks, export account data, sign out, or delete the account.

Core public endpoints include `/health`, `/api/currencies`, `/api/latest-rate`, `/api/latest-rates`, `/api/data-freshness`, `/api/history`, `/api/model-metadata`, `/api/predict-risk`, and `/api/export-excel`. Account routes under `/api` cover OTP authentication, session management, checks, account export, and account deletion.

## Current maintenance workflow

Refresh the official histories by keeping the original BNR workbooks unchanged and adding supplementary exports to `data/raw/` as `<CURRENCY> additional YYYY-MM-DD to YYYY-MM-DD.xlsx`. The synchronization pipeline validates and merges them chronologically. Then run:

```bash
python scripts/sync_multicurrency_rates.py
python scripts/train_multicurrency_models.py
python -m unittest discover -s tests -v
```

Run the application locally with `python run_backend.py`. The browser interface is served at `http://127.0.0.1:8000`, and interactive API documentation is available at `http://127.0.0.1:8000/docs`.

## Relationship to the remaining notebooks

This overview reflects the **current multicurrency application**. Notebooks `01` through `05` preserve the earlier guided USD/RWF data-science workflow for explanation and academic traceability; they do not create the active multicurrency production artifacts. Use the synchronization and multicurrency training scripts for the current system.

Detailed current evidence is maintained in `reports/multicurrency_model_evaluation.md`, `reports/testing_and_backtesting_report.md`, and `reports/README.md`.